## 데이터 수집 자동화

## 다나와 상품정보 스크래핑

#### 자동 로그인
- 패스워드 숨기기

#### 1. setx 명령어로 시스템에 저장방법
- 패스워드 숨기기
- setx 명령어로 시스템에 저장방법
- powershell

```powershell
> setx DANAWA_PASSWORD "비밀번호"

성공 : 지정한 값을 저장했습니다.

> setx DANAWA_ID "아이디"


성공 : 지정한 값을 저장했습니다.
```

- sysdm.cpl로 시스템 정보에서 환경변수 user 리스트에서 오픈 및 삭제

##### 2. keyring 으로 윈도우 자격증명 저장소

- setx 와 달리 아예 오픈 안됨
- 패키지 설치

In [1]:
!pip install keyring

Defaulting to user installation because normal site-packages is not writeable

   --------------------------------- ------ 5/6 [keyring]
   ---------------------------------------- 6/6 [keyring]



In [2]:
import keyring
from getpass import getpass

In [3]:
username = input('다나와 아이디')
password = getpass('다나와 비밀번호')

# 아이디 저장
keyring.set_password('danawa.com', 'user_id', username)
# 패스워드
keyring.set_password('danawa.com', username, password)

print('아이디/패스워드 저장 완료')

아이디/패스워드 저장 완료


In [4]:
user_id = keyring.get_password('danawa.com', 'user_id')
user_id

'personar'

In [5]:
password = keyring.get_password('danawa.com', user_id)
password is not None

True

- 패스워드 오픈 안되고 저장

In [9]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import os

In [32]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://prod.danawa.com/')

wait.until(
    EC.presence_of_element_located((By.ID, 'danawa_footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="4a837b98799daaf1b11a013a315cb64c", element="f.C2297E2F3C8E031CF24B6770016CF76C.d.D35F5789A56D4D2910CBFE4EAD601C78.e.73")>

In [11]:
# 로그인 링크 확인 후 클릭
driver.find_element(By.LINK_TEXT, '로그인').click()

In [12]:
# 로그인하기

id_input = driver.find_element(By.CSS_SELECTOR, 'input[type="text"]')
id_input.send_keys(user_id)

pass_input = driver.find_element(By.CSS_SELECTOR, 'input[type="password"]')
pass_input.send_keys(password)

In [13]:
login_btn = driver.find_element(By.CLASS_NAME, 'btn_login')
login_btn.click()

#### 동적 페이지 스크래핑

- https://prod.danawa.com/list/?cate=11255834&15main_11_02


In [33]:
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="4a837b98799daaf1b11a013a315cb64c", element="f.C2297E2F3C8E031CF24B6770016CF76C.d.9B58229EF37299044FED2E14D2E8C07C.e.1909")>

- select 태그 페이지 내에 1개만 존재
- 검색결과 90개로 조정

In [34]:
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

wait.until(
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="4a837b98799daaf1b11a013a315cb64c", element="f.C2297E2F3C8E031CF24B6770016CF76C.d.9B58229EF37299044FED2E14D2E8C07C.e.1909")>

In [35]:
products = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]')
len(products)

90

In [42]:
product = products[0]
product.text

'1\n영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060\n5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / AMD / 라이젠 5000시리즈 / 라이젠5 / 버미어 / DDR4 / 16GB / M.2 / 256GB / NVIDIA / 그래픽 메모리: 8GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / LED쿨러 / 용도: 게임용\n혜택 최저가\n1,125,110원\n[옥션] 삼성카드\n/\n무이자 최대 24개월\n기획전\n성능도 가격도 GOOD! 조립PC 인기상품 추천\n21.03. 등록\n브랜드로그\n의견 70\n4.8\n리뷰수\n(230)\n관심\n대량구매\n2위\n16GB, M.2 256GB\n1,209,640\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.\n16GB, M.2 512GB\n1,259,670\n원\n5몰\nVS상품비교에 추가\n1위\n32GB, M.2 1TB\n1,539,640\n원\n5몰\nVS상품비교에 추가'

In [ ]:
## 1번 인덱스 값에서 제품명, 제품스펙, 2번 인덱스 값에서 가격
product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')

[<selenium.webdriver.remote.webelement.WebElement (session="4a837b98799daaf1b11a013a315cb64c", element="f.C2297E2F3C8E031CF24B6770016CF76C.d.9B58229EF37299044FED2E14D2E8C07C.e.9289")>,
 <selenium.webdriver.remote.webelement.WebElement (session="4a837b98799daaf1b11a013a315cb64c", element="f.C2297E2F3C8E031CF24B6770016CF76C.d.9B58229EF37299044FED2E14D2E8C07C.e.9298")>,
 <selenium.webdriver.remote.webelement.WebElement (session="4a837b98799daaf1b11a013a315cb64c", element="f.C2297E2F3C8E031CF24B6770016CF76C.d.9B58229EF37299044FED2E14D2E8C07C.e.9412")>]

In [44]:
computer_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[1]
computer_info.text

'1\n영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060\n5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / AMD / 라이젠 5000시리즈 / 라이젠5 / 버미어 / DDR4 / 16GB / M.2 / 256GB / NVIDIA / 그래픽 메모리: 8GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / LED쿨러 / 용도: 게임용\n혜택 최저가\n1,125,110원\n[옥션] 삼성카드\n/\n무이자 최대 24개월\n기획전\n성능도 가격도 GOOD! 조립PC 인기상품 추천\n21.03. 등록\n브랜드로그\n의견 70\n4.8\n리뷰수\n(230)\n관심\n대량구매'

In [ ]:
# 컴퓨터 이름 추출
computer_info.find_element(By.CSS_SELECTOR, 'div a'). text

'영웅컴퓨터 영웅 875 게이밍울트라560X R5 5600 RTX5060'

In [53]:
# 컴퓨터 사양 추출
specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
' / '.join(spec.text for spec in specs)

'5600 / RTX 5060 / (AMD) B550 / OS미포함 / 600W / AMD / 라이젠 5000시리즈 / 라이젠5 / 버미어 / DDR4 / 16GB / M.2 / 256GB / NVIDIA / 그래픽 메모리: 8GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / LED쿨러 / 용도: 게임용'

In [46]:
price_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[2]
price_info.text

'2위\n16GB, M.2 256GB\n1,209,640\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.\n16GB, M.2 512GB\n1,259,670\n원\n5몰\nVS상품비교에 추가\n1위\n32GB, M.2 1TB\n1,539,640\n원\n5몰\nVS상품비교에 추가'

In [ ]:
# 가격정보 추출
price_info.find_element(By.CSS_SELECTOR, 'li').text

'2위\n16GB, M.2 256GB\n1,209,640\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'